In [1]:
import os
import numpy as np
import pandas as pd
import networkx as nx


In [ ]:

RUTA_CSV = "Data_Set/Red_Quimica_Comercio_Importaciones.csv"
CARPETA_SALIDA = "Data_Set"
CALCULAR_ARISTAS = False   # True = además exporta medidas locales de cada arista (más lento)
os.makedirs(CARPETA_SALIDA, exist_ok=True)

# ------------------------------------------------------------------
# 1. Construcción del grafo dirigido (Origen -> Destino)
# ------------------------------------------------------------------
df = pd.read_csv(RUTA_CSV)
G = nx.from_pandas_edgelist(
    df, source="Origen", target="Destino",
    edge_attr=["Peso_Volumen", "Resistencia_Ruta"],
    create_using=nx.DiGraph(),
)
G.remove_edges_from(nx.selfloop_edges(G))
n, m = G.number_of_nodes(), G.number_of_edges()
print(f"Nodos: {n} | Aristas: {m}")

# Resistencia_Ruta (= 1/volumen) se usa como DISTANCIA en las medidas de caminos.
# Peso_Volumen se usa como PESO (importancia) en fuerzas y PageRank.
DIST = "Resistencia_Ruta"
PESO = "Peso_Volumen"

# ------------------------------------------------------------------
# 2. Medidas locales de cada NODO
# ------------------------------------------------------------------
print("Calculando medidas locales de nodos...")
nodos = pd.DataFrame(index=pd.Index(list(G.nodes()), name="pais"))

nodos["grado_entrada"] = pd.Series(dict(G.in_degree()))
nodos["grado_salida"] = pd.Series(dict(G.out_degree()))
nodos["grado_total"] = nodos["grado_entrada"] + nodos["grado_salida"]

# Fuerza: suma de volúmenes (Origen exporta -> Destino importa)
nodos["volumen_exportado"] = pd.Series(dict(G.out_degree(weight=PESO)))
nodos["volumen_importado"] = pd.Series(dict(G.in_degree(weight=PESO)))
nodos["volumen_total"] = nodos["volumen_exportado"] + nodos["volumen_importado"]
nodos["balanza_comercial"] = nodos["volumen_exportado"] - nodos["volumen_importado"]

nodos["centralidad_grado"] = pd.Series(nx.degree_centrality(G))
nodos["centralidad_grado_entrada"] = pd.Series(nx.in_degree_centrality(G))
nodos["centralidad_grado_salida"] = pd.Series(nx.out_degree_centrality(G))

# Intermediación: con distancia = Resistencia_Ruta (1/volumen) los caminos "más cortos"
# son las rutas de mayor volumen; la versión en saltos ignora los volúmenes.
nodos["intermediacion"] = pd.Series(nx.betweenness_centrality(G, weight=DIST))
nodos["intermediacion_saltos"] = pd.Series(nx.betweenness_centrality(G))

# Cercanía en saltos (la métrica principal). La versión ponderada con 1/volumen queda
# dominada por unas pocas rutas de volumen minúsculo y casi no distingue entre países.
nodos["cercania_entrada"] = pd.Series(nx.closeness_centrality(G))
nodos["cercania_salida"] = pd.Series(nx.closeness_centrality(G.reverse(copy=False)))
nodos["cercania_entrada_ponderada"] = pd.Series(nx.closeness_centrality(G, distance=DIST))
nodos["cercania_salida_ponderada"] = pd.Series(nx.closeness_centrality(G.reverse(copy=False), distance=DIST))

nodos["pagerank"] = pd.Series(nx.pagerank(G, weight=PESO))

hubs, autoridades = nx.hits(G, max_iter=1000)   # sin pesos
nodos["hits_hub"] = pd.Series(hubs)
nodos["hits_autoridad"] = pd.Series(autoridades)

nodos["clustering"] = pd.Series(nx.clustering(G))   # versión dirigida, sin pesos

# Reciprocidad local: fracción de socios con los que el comercio va en ambos sentidos
recip = {}
for v in G.nodes():
    vecinos = set(G.successors(v)) | set(G.predecessors(v))
    mutuos = set(G.successors(v)) & set(G.predecessors(v))
    recip[v] = len(mutuos) / len(vecinos) if vecinos else 0.0
nodos["reciprocidad_local"] = pd.Series(recip)

nodos = nodos.sort_values("grado_total", ascending=False)
nodos.to_csv(os.path.join(CARPETA_SALIDA, "centralidades_nodos_imp.csv"))

# ------------------------------------------------------------------
# 3. Medidas locales de cada ARISTA
# ------------------------------------------------------------------
if CALCULAR_ARISTAS:
    print("Calculando medidas locales de aristas...")
    eb = nx.edge_betweenness_centrality(G, weight=DIST)

    filas = []
    for u, v, d in G.edges(data=True):
        reciproca = G.has_edge(v, u)
        filas.append({
            "origen": u,
            "destino": v,
            "peso_volumen": d[PESO],
            "resistencia_ruta": d[DIST],
            "intermediacion_arista": eb[(u, v)],
            "es_reciproca": reciproca,
            "peso_sentido_inverso": G[v][u][PESO] if reciproca else np.nan,
            "participacion_en_exportaciones_origen": d[PESO] / nodos.loc[u, "volumen_exportado"],
            "participacion_en_importaciones_destino": d[PESO] / nodos.loc[v, "volumen_importado"],
            "grado_origen": nodos.loc[u, "grado_total"],
            "grado_destino": nodos.loc[v, "grado_total"],
        })
    aristas = pd.DataFrame(filas).sort_values("peso_volumen", ascending=False)
    aristas.to_csv(os.path.join(CARPETA_SALIDA, "medidas_locales_aristas_imp.csv"), index=False)

# ------------------------------------------------------------------
# 4. Medidas globales
# ------------------------------------------------------------------
print("Calculando medidas globales...")
g_tot = nodos["grado_total"]

# Proyección no dirigida (suma de pesos en pares recíprocos) para transitividad y comunidades
U = nx.Graph()
U.add_nodes_from(G.nodes())
for u, v, d in G.edges(data=True):
    if U.has_edge(u, v):
        U[u][v]["weight"] += d[PESO]
    else:
        U.add_edge(u, v, weight=d[PESO])

# Caminos: solo tienen sentido dentro de la mayor componente fuertemente conexa
scc = max(nx.strongly_connected_components(G), key=len)
Gs = G.subgraph(scc).copy()

comunidades = nx.community.louvain_communities(U, weight="weight", seed=42)
modularidad = nx.community.modularity(U, comunidades, weight="weight")

globales = [
    ("n_nodos", n, "Número de países (nodos)"),
    ("n_aristas", m, "Número de relaciones comerciales dirigidas (aristas)"),
    ("densidad", nx.density(G), "Fracción de aristas posibles que existen"),
    ("reciprocidad", nx.reciprocity(G), "Fracción de aristas con su par en sentido contrario"),
    ("componentes_debiles", nx.number_weakly_connected_components(G), "Componentes débilmente conexas"),
    ("componentes_fuertes", nx.number_strongly_connected_components(G), "Componentes fuertemente conexas"),
    ("tam_mayor_comp_fuerte", len(scc), "Nodos en la mayor componente fuertemente conexa"),
    ("grado_entrada_promedio", m / n, "Grado de entrada promedio (= grado de salida promedio)"),
    ("grado_total_promedio", g_tot.mean(), "Grado total promedio (entrada + salida)"),
    ("grado_total_mediana", g_tot.median(), "Mediana del grado total"),
    ("grado_total_desv_est", g_tot.std(), "Desviación estándar del grado total"),
    ("grado_total_max", g_tot.max(), "Grado total máximo"),
    ("grado_total_min", g_tot.min(), "Grado total mínimo"),
    ("grado_total_asimetria", g_tot.skew(), "Asimetría de la distribución de grado (>0: cola derecha)"),
    ("asortatividad_grado", nx.degree_assortativity_coefficient(G), "Correlación de grado entre extremos de una arista"),
    ("clustering_promedio", nx.average_clustering(G), "Agrupamiento promedio (versión dirigida, sin pesos)"),
    ("transitividad", nx.transitivity(U), "Transitividad de la proyección no dirigida"),
    ("camino_promedio", nx.average_shortest_path_length(Gs), "Camino más corto promedio en saltos (mayor comp. fuerte)"),
    ("camino_promedio_ponderado", nx.average_shortest_path_length(Gs, weight=DIST), "Camino más corto promedio usando Resistencia_Ruta como distancia"),
    ("diametro", nx.diameter(Gs), "Diámetro en saltos (mayor comp. fuerte)"),
    ("n_comunidades", len(comunidades), "Comunidades detectadas con Louvain (proyección no dirigida)"),
    ("modularidad", modularidad, "Modularidad de la partición de Louvain"),
]
pd.DataFrame(globales, columns=["metrica", "valor", "descripcion"]).to_csv(
    os.path.join(CARPETA_SALIDA, "medidas_globales_imp.csv"), index=False
)

print(f"Listo. Archivos guardados en '{CARPETA_SALIDA}/':")
print(" - centralidades_nodos_imp.csv")
if CALCULAR_ARISTAS:
    print(" - medidas_locales_aristas_imp.csv")
print(" - medidas_globales_imp.csv")

Nodos: 229 | Aristas: 13492
Calculando medidas locales de nodos...
Calculando medidas globales...
Listo. Archivos guardados en 'Data_Set/':
 - centralidades_nodos_imp.csv
 - medidas_globales_imp.csv
